In [ ]:
# Tính chỉ số hàng ngày theo chuỗi
series = (
    df.groupby('id')
      .agg(zero_rate=('y', lambda s: (s == 0).mean()),
           avg_sales=('y', 'mean'))
      .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(series['zero_rate'], bins=30, color='steelblue', edgecolor='black')
ax.axvline(series['zero_rate'].median(), color='red', linestyle='--', label='median')
ax.set_title('Hình 2.1. Phân bố tỷ lệ ngày bán bằng 0 theo chuỗi')
ax.set_xlabel('Tỷ lệ ngày bằng 0')
ax.set_ylabel('Số chuỗi')
ax.legend()
fig.tight_layout()
fig.savefig(root / 'results/figures/01_zero_rate_hist.png', dpi=200)
plt.close(fig)

# Doanh số trung bình theo thứ trong tuần
weekday = df.groupby('weekday')['y'].mean().reindex(['Saturday','Sunday','Monday','Tuesday','Wednesday','Thursday','Friday'])
fig, ax = plt.subplots(figsize=(8, 5))
weekday.plot(kind='bar', ax=ax, color='darkorange')
ax.set_title('Hình 2.2. Doanh số trung bình theo thứ trong tuần')
ax.set_xlabel('Thứ trong tuần')
ax.set_ylabel('Doanh số trung bình')
fig.tight_layout()
fig.savefig(root / 'results/figures/02_sales_by_weekday.png', dpi=200)
plt.close(fig)

# Doanh số trung bình theo tháng
monthly = df.groupby('month')['y'].mean()
fig, ax = plt.subplots(figsize=(8, 5))
monthly.plot(kind='bar', ax=ax, color='seagreen')
ax.set_title('Hình 2.3. Doanh số trung bình theo tháng')
ax.set_xlabel('Tháng')
ax.set_ylabel('Doanh số trung bình')
fig.tight_layout()
fig.savefig(root / 'results/figures/03_sales_by_month.png', dpi=200)
plt.close(fig)

# So sánh SNAP và sự kiện
snap = df.groupby('snap_CA')['y'].mean().rename({0: 'Khong SNAP', 1: 'Co SNAP'})
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(snap.index, snap.values, color=['#7aa6c2', '#f4a261'])
ax.set_title('Hình 2.4. Doanh số theo trạng thái SNAP')
ax.set_xlabel('Trạng thái SNAP')
ax.set_ylabel('Doanh số trung bình')
fig.tight_layout()
fig.savefig(root / 'results/figures/04_snap_compare.png', dpi=200)
plt.close(fig)

# Sự kiện
has_event = df['event_name_1'].notna().astype(int)
df['has_event'] = has_event
fig, ax = plt.subplots(figsize=(6, 5))
by_event = df.groupby('has_event')['y'].mean().rename({0: 'Khong su kien', 1: 'Co su kien'})
ax.bar(by_event.index.map({0:'Khong su kien',1:'Co su kien'}), by_event.values, color=['#8ecae6', '#ffb703'])
ax.set_title('Hình 2.5. Doanh số khi có sự kiện vs khong có sự kiện')
ax.set_xlabel('Trạng thái sự kiện')
ax.set_ylabel('Doanh số trung bình')
fig.tight_layout()
fig.savefig(root / 'results/figures/05_event_compare.png', dpi=200)
plt.close(fig)

# Giá của 5 mặt hàng ngẫu nhiên
rng = np.random.default_rng(42)
items = df['item_id'].drop_duplicates().sample(5, random_state=42).tolist()
price_df = df[df['item_id'].isin(items)][['item_id','date','sell_price']].copy()
fig, ax = plt.subplots(figsize=(10, 6))
for item in items:
    sub = price_df[price_df['item_id'] == item].sort_values('date')
    ax.plot(sub['date'], sub['sell_price'], label=item)
ax.set_title('Hình 2.6. Động thái giá của 5 mặt hàng ngẫu nhiên')
ax.set_xlabel('Ngày')
ax.set_ylabel('Giá bán')
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(root / 'results/figures/06_price_trend_sample.png', dpi=200)
plt.close(fig)

# Tổng doanh số theo ngày
agg = df.groupby('date')['y'].sum().reset_index()
agg['ma_28'] = agg['y'].rolling(28, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(agg['date'], agg['y'], label='Doanh so ngay', color='gray', alpha=0.5)
ax.plot(agg['date'], agg['ma_28'], label='Trung binh 28 ngay', color='darkred', linewidth=2)
ax.set_title('Hình 2.7. Tổng doanh số cả nhóm hàng theo ngày')
ax.set_xlabel('Ngày')
ax.set_ylabel('Tổng doanh số')
ax.legend()
fig.tight_layout()
fig.savefig(root / 'results/figures/07_daily_total_sales.png', dpi=200)
plt.close(fig)

print('Da luu hinh anh EDA vao results/figures/')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

root = Path('..').resolve()
df = pd.read_parquet(root / 'data/processed/ca1_foods3.parquet')
print(df.shape)
print(df[['id','date','y','sell_price','snap_CA']].head())


# Bước 2: EDA cho dữ liệu M5 CA_1 x FOODS_3

Tài liệu này phân tích chuỗi bán hàng, tính chất ngày SNAP, ảnh hưởng sự kiện, biến động giá và xu hướng tổng doanh số.
